# FUNCTION VECTORS ACROSS PARTS OF SPEECH #

## Загрузка Датасета ##

In [1]:
!git clone https://github.com/nguyenkh/AntSynNET.git

fatal: destination path 'AntSynNET' already exists and is not an empty directory.


In [2]:
import pandas as pd

In [3]:
possible_pos = ['adjective', 'noun', 'verb']
possible_ext = ['val', 'test', 'train']
RAW = {
    'SYN': {},
    'ANT': {}
}

for pos in possible_pos:
  df1 = pd.read_csv(f'/content/AntSynNET/dataset/{pos}-pairs.train', sep='\t', header=None, names=['w1', 'w2', 'result'])
  df2 = pd.read_csv(f'/content/AntSynNET/dataset/{pos}-pairs.test', sep='\t', header=None, names=['w1', 'w2', 'result'])
  df3 = pd.read_csv(f'/content/AntSynNET/dataset/{pos}-pairs.val', sep='\t', header=None, names=['w1', 'w2', 'result'])
  df = pd.concat([df1, df2, df3])
  RAW['SYN'][pos] = df[df['result'] == 0].drop(columns=['result'])
  RAW['ANT'][pos] = df[df['result'] == 1].drop(columns=['result'])

In [4]:
DataSets = {}
visual_data = [['type'] + possible_pos]
for type_str in RAW.keys():
  local_visual = [type_str]
  for pos in possible_pos:
    df = RAW[type_str][pos]
    data = []
    for i in range(0, len(df)//11*11, 11):
      p_i = ''
      for j in range(i, i+11):
        p_i += f"Q:{df.iloc[j]['w1']}"
        if j != i + 10:
          p_i += f"\nA:{df.iloc[j]['w2']}\n\n"
        else:
          p_i += '\nA:'
          y = df.iloc[j]['w2']
      data.append([p_i, y])
    DataSets[f'{type_str}_{pos}'] = pd.DataFrame(data, columns=['X', 'Y'])
    local_visual.append(len(data))
  visual_data.append(local_visual)

In [5]:
pd.DataFrame(visual_data[1:], columns=visual_data[0])

,type,adjective,noun,verb
0,SYN,361,184,164
1,ANT,361,184,164


## Загрузка GPT-2 ##

In [6]:
!pip install transformer_lens

In [7]:
from transformer_lens import HookedTransformer

model = HookedTransformer.from_pretrained("gpt2", device="cuda")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Loaded pretrained model gpt2 into HookedTransformer


In [8]:
from tqdm import tqdm
import torch

In [9]:
FV_res = {}

In [10]:
for dt_key in tqdm(DataSets.keys()):
  dataset = DataSets[dt_key]
  promts = dataset['X'].tolist()
  all_res = []
  for promt in promts:
    outp = model.run_with_cache(promt)
    cache = outp[-1]
    res_per_promt = []
    for li in range(model.cfg.n_layers):
      reskey = f'blocks.{li}.hook_attn_out'
      res = cache[reskey][0, -1, :]
      res_per_promt.append(res)
    vector = torch.stack(res_per_promt).sum(dim=0)
    all_res.append(vector)
  fv = torch.stack(all_res).mean(dim=0)
  FV_res[dt_key] = fv

100%|██████████| 6/6 [01:38<00:00, 16.43s/it]


In [11]:
import numpy as np

In [12]:
def cos_sim(vec1, vec2):
  return (vec1 @ vec2) / (vec1.norm() * vec2.norm())

In [18]:
keys = list(FV_res.keys())

data = []
for i in range(len(keys)):
  for j in range(i+1, len(keys)):
    data.append([keys[i], keys[j],  float(cos_sim(FV_res[keys[i]], FV_res[keys[j]]))])
dfFV = pd.DataFrame(data, columns=['FV1', 'FV2', 'cos_similarity'])

In [19]:
dfFV

,FV1,FV2,cos_similarity
0,SYN_adjective,SYN_noun,0.993078
1,SYN_adjective,SYN_verb,0.990863
2,SYN_adjective,ANT_adjective,0.998309
3,SYN_adjective,ANT_noun,0.994423
4,SYN_adjective,ANT_verb,0.990174
5,SYN_noun,SYN_verb,0.994342
6,SYN_noun,ANT_adjective,0.993069
7,SYN_noun,ANT_noun,0.997577
8,SYN_noun,ANT_verb,0.992071
9,SYN_verb,ANT_adjective,0.990163


In [20]:
dfFV['cos_similarity_centered'] =  dfFV['cos_similarity'] - dfFV['cos_similarity'].mean()

In [28]:
dfFV.sort_values('cos_similarity_centered')

,FV1,FV2,cos_similarity,cos_similarity_centered
9,SYN_verb,ANT_adjective,0.990163,-0.003579
4,SYN_adjective,ANT_verb,0.990174,-0.003569
13,ANT_adjective,ANT_verb,0.990629,-0.003114
1,SYN_adjective,SYN_verb,0.990863,-0.002880
8,SYN_noun,ANT_verb,0.992071,-0.001671
6,SYN_noun,ANT_adjective,0.993069,-0.000673
0,SYN_adjective,SYN_noun,0.993078,-0.000665
14,ANT_noun,ANT_verb,0.993313,-0.000429
10,SYN_verb,ANT_noun,0.993361,-0.000381
5,SYN_noun,SYN_verb,0.994342,0.000600


## Каузальная проверка ##

In [66]:
cas_checks = list(FV_res.keys())
columns = ['Case', 'No FV', 'FV-mean-0%', 'FV-mean-50%', 'FV-mean-100%', 'FV-type', 'FV-exact'] + cas_checks
data = []

In [34]:
FV_mean = sum(list(FV_res.values()))
FV_mean = FV_mean/len(FV_mean)
FV_SYN = sum([FV_res[k] for k in FV_res.keys() if 'SYN' in k])
FV_SYN = FV_SYN/len(FV_SYN)
FV_ANT = sum([FV_res[k] for k in FV_res.keys() if 'ANT' in k])
FV_ANT = FV_ANT/len(FV_ANT)

In [64]:
def add_info_fv(fv):
  def add_info(activation, hook):
    activation[0, -1, :] += fv
    return activation
  return add_info

In [40]:
model.blocks

ModuleList(
  (0-11): 12 x TransformerBlock(
    (ln1): LayerNormPre(
      (hook_scale): HookPoint()
      (hook_normalized): HookPoint()
    )
    (ln2): LayerNormPre(
      (hook_scale): HookPoint()
      (hook_normalized): HookPoint()
    )
    (attn): Attention(
      (hook_k): HookPoint()
      (hook_q): HookPoint()
      (hook_v): HookPoint()
      (hook_z): HookPoint()
      (hook_attn_scores): HookPoint()
      (hook_pattern): HookPoint()
      (hook_result): HookPoint()
    )
    (mlp): MLP(
      (hook_pre): HookPoint()
      (hook_post): HookPoint()
    )
    (hook_attn_in): HookPoint()
    (hook_q_input): HookPoint()
    (hook_k_input): HookPoint()
    (hook_v_input): HookPoint()
    (hook_mlp_in): HookPoint()
    (hook_attn_out): HookPoint()
    (hook_mlp_out): HookPoint()
    (hook_resid_pre): HookPoint()
    (hook_resid_mid): HookPoint()
    (hook_resid_post): HookPoint()
  )
)

In [42]:
torch.cuda.empty_cache()

In [68]:
with torch.no_grad():
  for dt_key in tqdm(DataSets.keys()):
    dataset = DataSets[dt_key]
    promts = dataset['X'].tolist()[:50]
    anwers = [str(x) for x in dataset['Y'].tolist()[:50]]
    res = [0] * len(columns)
    res[0] = dt_key
    i = -1
    for promt in promts:
      i += 1
      promt = promt.split()[-2] + '\n' + promt.split()[-1]
      # No FV
      logits = model(promt)
      probs = logits[0, -1, :].softmax(dim=-1)
      prob = probs[model.tokenizer.encode(anwers[i])[0]].item()
      res[1] += prob
      # FV-mean-0%
      model.reset_hooks()
      torch.cuda.empty_cache()
      model.add_hook("blocks.0.hook_resid_post", add_info_fv(FV_mean))
      logits = model(promt)
      probs = logits[0, -1, :].softmax(dim=-1)
      prob = probs[model.tokenizer.encode(anwers[i])[0]].item()
      res[2] += prob
      # FV-mean-50%
      model.reset_hooks()
      torch.cuda.empty_cache()
      model.add_hook("blocks.5.hook_resid_post", add_info_fv(FV_mean))
      logits = model(promt)
      probs = logits[0, -1, :].softmax(dim=-1)
      prob = probs[model.tokenizer.encode(anwers[i])[0]].item()
      res[3] += prob
      # FV-mean-100%
      model.reset_hooks()
      torch.cuda.empty_cache()
      model.add_hook("blocks.11.hook_resid_post", add_info_fv(FV_mean))
      logits = model(promt)
      probs = logits[0, -1, :].softmax(dim=-1)
      prob = probs[model.tokenizer.encode(anwers[i])[0]].item()
      res[4] += prob
      # FV-type
      model.reset_hooks()
      torch.cuda.empty_cache()
      model.add_hook("blocks.11.hook_resid_post", add_info_fv((FV_SYN if 'SYN' in dt_key else FV_ANT)))
      logits = model(promt)
      probs = logits[0, -1, :].softmax(dim=-1)
      prob = probs[model.tokenizer.encode(anwers[i])[0]].item()
      res[5] += prob
      # FV-excact
      model.reset_hooks()
      torch.cuda.empty_cache()
      model.add_hook("blocks.11.hook_resid_post", add_info_fv(FV_res[dt_key]))
      logits = model(promt)
      probs = logits[0, -1, :].softmax(dim=-1)
      prob = probs[model.tokenizer.encode(anwers[i])[0]].item()
      res[6] += prob
      # ohter FV
      for j in range(7, len(columns)):
        model.reset_hooks()
        torch.cuda.empty_cache()
        model.add_hook("blocks.11.hook_resid_post", add_info_fv(FV_res[columns[j]]))
        logits = model(promt)
        probs = logits[0, -1, :].softmax(dim=-1)
        prob = probs[model.tokenizer.encode(anwers[i])[0]].item()
        res[j] += prob
    for i in range(1, len(res)):
      res[i] /= len(res)
    data.append(res)


100%|██████████| 6/6 [03:30<00:00, 35.15s/it]


In [69]:
dfCasual = pd.DataFrame(data, columns=columns)

In [71]:
dfCasual

,Case,No FV,FV-mean-0%,FV-mean-50%,FV-mean-100%,FV-type,FV-exact,SYN_adjective,SYN_noun,SYN_verb,ANT_adjective,ANT_noun,ANT_verb
0,SYN_adjective,0.012640,0.018754,0.018409,0.019198,0.019230,0.014318,0.014318,0.013096,0.012427,0.013338,0.013204,0.012640
1,SYN_noun,0.012852,0.028883,0.028650,0.029752,0.029868,0.014956,0.014511,0.014956,0.013536,0.013521,0.013735,0.012852
2,SYN_verb,0.010127,0.012008,0.010063,0.012533,0.012531,0.009025,0.010478,0.008833,0.009025,0.009268,0.008841,0.010127
3,ANT_adjective,0.006194,0.006432,0.006975,0.006697,0.006710,0.006601,0.006779,0.005786,0.005786,0.006601,0.005819,0.006194
4,ANT_noun,0.011608,0.030658,0.031961,0.030956,0.031101,0.011301,0.012791,0.012125,0.011579,0.011153,0.011301,0.011608
5,ANT_verb,0.005493,0.003648,0.003633,0.003725,0.003715,0.005493,0.005432,0.004971,0.005065,0.005135,0.005116,0.005493


In [70]:
baseline_imporvement = ((dfCasual['FV-mean-100%'] - dfCasual['No FV'])/dfCasual['No FV']).mean()
type_imporvement = ((dfCasual['FV-type'] - dfCasual['No FV'])/dfCasual['No FV']).mean()
exact_fv_imporvement = ((dfCasual['FV-exact'] - dfCasual['No FV'])/dfCasual['No FV']).mean()
print(f'{baseline_imporvement*100//1}% {type_imporvement*100//1}% {exact_fv_imporvement*100//1}%')

58.0% 58.0% 3.0%
